Web Scraping

In [ ]:
import requests                     # hacer peticiones HTTP
from bs4 import BeautifulSoup       # analizar el contenido HTML (parseador)
import urllib.parse                 # obtener rutas de las url

#parser: es un analizador de texto que convierte el texto html en un objeto de Python que se puede manipular 

url = "https://books.toscrape.com/catalogue/category/books_1/index.html"    #html que se va a analizar

def extraer_html(url):
    respuesta = requests.get(url)                                                #extraer el contenido del html
    contenido_html = respuesta.text

    sopa = BeautifulSoup(contenido_html, "html.parser")  #analizar el contenido html
    print(respuesta.status_code)
    return sopa

def extraer_categorias(url):
    sopa = extraer_html(url)
    categorias = {} #diccionario para almacenar las categorias y sus enlaces
    etiqueta_categoria = sopa.find("div", class_="side_categories")  #buscar el div con la clase side_categories
    enlaces = etiqueta_categoria.find_all("a")

    for categoria in enlaces[1:]:  #omitir el primer enlace que es "All categories"
        nombre_categoria = categoria.text.strip()
        link_categoria = categoria["href"]
        link_completo = urllib.parse.urljoin(url, link_categoria) #obtener la url completa de la categoria
        
        categorias[nombre_categoria] = link_completo
    return categorias
categorias = extraer_categorias(url)

def extraer_libros(url_categoria, nombre_categoria):
    libros = []
    url_actual = url_categoria

    while True :
        sopa = extraer_html(url_actual)
        etiqueta_libro =sopa.select(".product_pod")

        for libro in etiqueta_libro:
            titulo = libro.select_one("h3 a")["title"]
            precio = libro.select_one(".price_color").get_text()
            rating = libro.select_one("p.star-rating")["class"][1]
            
            libros.append({"titulo": titulo,
                           "precio": precio,
                           "rating": rating,
                           "categoria": nombre_categoria})

        # Buscar el botón 'Next' para la paginación
        boton_siguiente = sopa.select_one("li.next a")
        if boton_siguiente:
            ruta_siguiente = boton_siguiente["href"]
            # Actualizamos la url para la siguiente iteración del while
            url = urllib.parse.urljoin(url, ruta_siguiente)
        else:
            break  # Si no hay botón 'Next', salimos del bucle    
    return libros

def extraer_libros(url_categoria, nombre_categoria):
    libros_categoria = []
    url_actual = url_categoria  # Empezamos por la URL de la categoría que recibimos

    while True:
        sopa = extraer_html(url_actual)
        
        # 1. Corregimos el selector: usamos "." para clase CSS
        etiquetas_libros = sopa.select(".product_pod")

        for libro in etiquetas_libros:
            # 2. Corregimos atributos y obtención de texto
            titulo = libro.select_one("h3 a")["title"] # Atributo 'title' (con una 't')
            
            # El precio es texto, no un atributo
            precio_texto = libro.select_one(".price_color").get_text()
            
            # El rating es la segunda clase de la etiqueta <p>
            rating = libro.select_one(".star-rating")["class"][1]
            
            # 3. Guardamos los datos incluyendo la CATEGORÍA ACTUAL
            libros_categoria.append({
                "titulo": titulo,
                "precio": precio_texto,
                "rating": rating,
                "categoria": nombre_categoria # Aquí vinculamos el libro con su categoría
            })

        # 4. Lógica de Paginación: buscamos el botón "next"
        boton_siguiente = sopa.select_one("li.next a")
        
        if boton_siguiente:
            # Construimos la URL de la siguiente página
            ruta_relativa = boton_siguiente["href"]
            url_actual = urllib.parse.urljoin(url_actual, ruta_relativa)
        else:
            # Si no hay más páginas, salimos del bucle while
            break
            
    return libros_categoria

def extraer_autores():
    print(categorias.keys)

extraer_autores()
       
todos_los_libros = []

# Recorremos el diccionario de categorías que ya tienes
for nombre, url_cat in categorias.items():
    print(f"Scrapeando categoría: {nombre}...")
    
    # Llamamos a la función corregida pasando la URL y el NOMBRE actual
    datos_libros = extraer_libros(url_cat, nombre)
    
    # Agregamos los libros encontrados a nuestra lista global
    todos_los_libros.extend(datos_libros)

print(f"Proceso finalizado. Total de libros extraídos: {len(todos_los_libros)}")

200
<built-in method keys of dict object at 0x00000282EFE52CC0>
Scrapeando categoría: Travel...
200
Scrapeando categoría: Mystery...
200
200
Scrapeando categoría: Historical Fiction...
200
200
Scrapeando categoría: Sequential Art...
200
200
200
200
Scrapeando categoría: Classics...
200
Scrapeando categoría: Philosophy...
200
Scrapeando categoría: Romance...
200


KeyboardInterrupt: 

In [ ]:
def extraer_autores():
    autores_links = []
    for link in categorias.values():
        autores_links.append(link)
    

extraer_autores()

['https://books.toscrape.com/catalogue/category/books/travel_2/index.html', 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html', 'https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html', 'https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html', 'https://books.toscrape.com/catalogue/category/books/classics_6/index.html', 'https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html', 'https://books.toscrape.com/catalogue/category/books/romance_8/index.html', 'https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html', 'https://books.toscrape.com/catalogue/category/books/fiction_10/index.html', 'https://books.toscrape.com/catalogue/category/books/childrens_11/index.html', 'https://books.toscrape.com/catalogue/category/books/religion_12/index.html', 'https://books.toscrape.com/catalogue/category/books/nonfiction_13/index.html', 'https://books.toscrape.com/catalogue/category/bo